In [1]:
import sys
import json
import pandas as pd

print(sys.executable)

model_spec = json.load(open("exports/model_features.json", encoding="utf-8"))
id_cols, target = model_spec["id_cols"], model_spec["target"]
baseline_features, extended_features = model_spec["baseline_features"], model_spec["extended_features"]

extended_data = pd.read_csv("exports/model_data.csv", parse_dates=["Date"], dtype={"season": str})
baseline_data = extended_data[id_cols + baseline_features + [target]]

print("baseline_data:", baseline_data.shape)
print("extended_data:", extended_data.shape)

c:\Users\maxma\Škola\5_semestr\ML2\ml2\.venv\Scripts\python.exe
baseline_data: (42552, 49)
extended_data: (42552, 96)


## 5. Train / test split

Before splitting the data, we inspect match counts, class balance (`over_2_5`), date ranges, and the COVID behind-closed-doors period (`closed_doors`) across all six seasons. We also inspect match counts per country in the final season (`2425`) to assess how noisy per-country accuracy estimates will be when pricing data per market.

Three constraints guide our split design:
1. **Strict chronological split:** Football results and rolling features evolve over time; splitting randomly would leak future matches into training.
2. **Elo tuning constraint:** Elo hyperparameters ($K$, home advantage, and division-change adjustments) were tuned on seasons `2020/21`–`2022/23`, so `2223` must stay in the training set to avoid validation or test leakage.
3. **Single-use test set & small-market noise:** Keeping `2324` as validation and `2425` as an untouched test season preserves a clean leak-free tuning set, though smaller markets in `2425` (233–341 matches) carry an approximate 95% confidence margin of error of ±5.3 to ±6.4 percentage points. Because this uncertainty can rival the accuracy gap between models, Section 7 will quantify per-country price reliability via bootstrap resampling rather than relying on point estimates alone.

In [7]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss

# 1. Inspect seasons: match counts, target balance, and COVID overlap
season_summary = (
    baseline_data.groupby("season")
    .agg(
        matches=(target, "size"),
        over_2_5_rate=(target, "mean"),
        closed_doors_matches=("closed_doors", "sum"),
        start_date=("Date", "min"),
        end_date=("Date", "max"),
    )
)
display(season_summary)

# 2. Inspect test-season sample sizes per country (Decision 3: small-market noise)
test_country_counts = (
    baseline_data.loc[baseline_data["season"] == "2425"]
    .groupby("country")
    .agg(
        matches_2425=(target, "size"),
        over_2_5_rate=(target, "mean"),
    )
    .sort_values("matches_2425", ascending=False)
)
# Approximate 95% binomial margin of error around p = 0.55: 1.96 * sqrt(0.55 * 0.45 / n)
test_country_counts["approx_95pct_moe_pp"] = (
    1.96 * np.sqrt(0.55 * 0.45 / test_country_counts["matches_2425"]) * 100
).round(1)
display(test_country_counts)

,matches,over_2_5_rate,closed_doors_matches,start_date,end_date
season,,,,,
1920,6453,0.503487,1040,2019-07-26,2020-08-07
2021,7169,0.475659,7169,2020-08-01,2021-05-30
2122,7313,0.490907,0,2021-07-23,2022-05-29
2223,7246,0.488821,0,2022-07-15,2023-06-07
2324,7245,0.531263,0,2023-07-28,2024-06-02
2425,7126,0.506876,0,2024-07-26,2025-06-01


,matches_2425,over_2_5_rate,approx_95pct_moe_pp
country,,,
england,2036,0.485756,2.2
spain,842,0.484561,3.4
scotland,768,0.518229,3.5
italy,760,0.469737,3.5
france,612,0.517974,3.9
germany,611,0.597381,3.9
turkey,341,0.551320,5.3
belgium,311,0.517685,5.5
netherlands,306,0.552288,5.6


### 5.1 Chronological split helper

We define a shared `split_by_season` helper so both the baseline and extended models use the exact same chronological train (`1920`–`2223`), validation (`2324`), and test (`2425`) boundaries, with built-in assertions preventing date overlap or moving `2223` out of training.

Because `1920` is a warm-up season (higher missingness in rolling form features and all Elo ratings starting at 1500) and `May 2020–June 2021` covers behind-closed-doors COVID matches with lower scoring rates, the helper supports testing alternative training windows while keeping `2324` and `2425` fixed. We never drop both `1920` and the COVID period simultaneously, as doing so would cut the training set down to ~14,600 matches. Any row filtering is done only on the post-feature datasets (`baseline_data` / `extended_data`), preserving full match history for later rolling features and Elo ratings.

In [3]:
TRAIN_SEASONS = ["1920", "2021", "2122", "2223"]
VAL_SEASONS = ["2324"]
TEST_SEASONS = ["2425"]


def split_by_season(
    df,
    train_seasons=TRAIN_SEASONS,
    val_seasons=VAL_SEASONS,
    test_seasons=TEST_SEASONS,
    drop_closed_doors=False,
):
    """Chronological train/val/test split. Optionally drops COVID matches from train only."""
    train = df[df["season"].isin(train_seasons)].copy()
    if drop_closed_doors:
        train = train[train["closed_doors"] == 0].copy()

    val = df[df["season"].isin(val_seasons)].copy()
    test = df[df["season"].isin(test_seasons)].copy()

    # Safety assertions: strict chronological order and no overlap
    assert train["Date"].max() < val["Date"].min(), "Train/val date overlap!"
    assert val["Date"].max() < test["Date"].min(), "Val/test date overlap!"
    assert "2223" in train_seasons, "2223 must remain in train (Elo tuning constraint)."

    return train, val, test


# Check training sizes for the candidate training-window options
for name, seasons, drop_cd in [
    ("All 1920-2223", ["1920", "2021", "2122", "2223"], False),
    ("Drop 1920 (2021-2223)", ["2021", "2122", "2223"], False),
    ("Drop closed_doors (keep 1920)", ["1920", "2021", "2122", "2223"], True),
]:
    tr, va, te = split_by_season(baseline_data, train_seasons=seasons, drop_closed_doors=drop_cd)
    print(f"{name:32s} | train: {len(tr):5d} | val: {len(va):4d} | test: {len(te):4d}")

All 1920-2223                    | train: 28181 | val: 7245 | test: 7126
Drop 1920 (2021-2223)            | train: 21728 | val: 7245 | test: 7126
Drop closed_doors (keep 1920)    | train: 19972 | val: 7245 | test: 7126


### 5.2 Shared evaluation function and prediction interface

All models (baseline and extended) are evaluated with the same `evaluate_predictions` function, which reports both overall and per-country metrics:
* **Accuracy** (at threshold 0.5, since classes are balanced at 49.9% positive): directly feeds $A_0$ and $A_1$ into the profit-pricing model in Section 7.
* **Log loss:** smoother and less noisy than accuracy, making it our primary criterion when comparing training-window variants and hyperparameters on `2324`.

We also define `build_predictions_df` to standardize per-match probability exports (`country`, `season`, `Date`, `HomeTeam`, `AwayTeam`, actual `over_2_5`, and predicted probabilities) for downstream pricing and bootstrap analysis. Finally, we evaluate a trivial constant-probability baseline (the training-set positive rate) on `2324` as a sanity-check floor.

In [8]:
def evaluate_predictions(df, y_prob, model_name="model", threshold=0.5):
    """
    Compute overall and per-country accuracy and log loss.

    Parameters
    ----------
    df : DataFrame containing at least 'country' and `target` ('over_2_5')
    y_prob : 1D array-like of predicted probabilities for class 1 (over 2.5)
    model_name : label for summary tables
    threshold : classification cutoff for accuracy (default 0.5)

    Returns
    -------
    overall : dict with overall metrics
    by_country : DataFrame with per-country metrics sorted by match count
    """
    y_true = df[target].to_numpy()
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= threshold).astype(int)

    overall = {
        "model": model_name,
        "matches": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "log_loss": log_loss(y_true, y_prob, labels=[0, 1]),
        "pred_over_rate": y_pred.mean(),
        "true_over_rate": y_true.mean(),
    }

    rows = []
    for country, idx in df.groupby("country").groups.items():
        pos = df.index.get_indexer(idx)
        yt = y_true[pos]
        yp = y_prob[pos]
        rows.append(
            {
                "country": country,
                "matches": len(yt),
                "accuracy": accuracy_score(yt, (yp >= threshold).astype(int)),
                "log_loss": log_loss(yt, yp, labels=[0, 1]),
            }
        )

    by_country = (
        pd.DataFrame(rows)
        .sort_values("matches", ascending=False)
        .reset_index(drop=True)
    )
    return overall, by_country


def build_predictions_df(df, prob_dict):
    """
    Create the standardized per-match prediction table for Person C.
    prob_dict: mapping of column_name -> array of predicted probabilities,
               e.g. {'prob_baseline_xgb': probs_xgb, 'prob_baseline_logreg': probs_lr}
    """
    out = df[["country", "season", "Date", "HomeTeam", "AwayTeam", target]].copy()
    for col_name, probs in prob_dict.items():
        out[col_name] = np.asarray(probs, dtype=float)
    return out


# Quick sanity check on the validation season using a trivial constant-probability baseline
train_df, val_df, test_df = split_by_season(baseline_data)
p_prior = np.full(len(val_df), train_df[target].mean())
naive_overall, naive_by_country = evaluate_predictions(val_df, p_prior, model_name="Naive (train prior)")
print(pd.DataFrame([naive_overall]).to_string(index=False))

              model  matches  accuracy  log_loss  pred_over_rate  true_over_rate
Naive (train prior)     7245  0.468737  0.694702             0.0        0.531263


## 6. Models

### 6.1 Baseline models

Baseline models use only the 41 `baseline_features` derived from the company's current basic match data (never `id_cols` and never `extended_features`). We build two model families:
* **XGBoost (primary baseline model):** handles missing values natively by learning optimal split directions at each node, so we leave `NaN` values intact.
* **Logistic Regression (benchmark):** tests whether gradient-boosted trees add meaningful value over a regularized linear model, and provides a shared pipeline that will also be reused unchanged on the extended features.

Before building the Logistic Regression pipeline, we inspect missing shares and upper-tail outliers strictly on the training set (`1920`–`2223`) so preprocessing choices are justified without touching validation or test data.

In [9]:
train_df, val_df, test_df = split_by_season(baseline_data)

stats = pd.DataFrame({
    "missing_pct": (train_df[baseline_features].isna().mean() * 100).round(2),
    "min": train_df[baseline_features].min(),
    "p99": train_df[baseline_features].quantile(0.99),
    "p995": train_df[baseline_features].quantile(0.995),
    "max": train_df[baseline_features].max(),
})

print("Top 10 baseline features by missing % (train only):")
display(stats.sort_values("missing_pct", ascending=False).head(10))

print("Baseline features with largest upper-tail jump (max / p995):")
tail_ratio = (stats["max"] / stats["p995"].replace(0, np.nan)).sort_values(ascending=False)
display(stats.loc[tail_ratio.head(8).index])

Top 10 baseline features by missing % (train only):


,missing_pct,min,p99,p995,max
home_venue_goals_for_5,4.74,0.000000,3.40,3.651669,5.20000
away_venue_goals_for_5,4.74,0.000000,2.80,3.000000,5.20000
home_venue_points_5,4.74,0.000000,3.00,3.000000,3.00000
away_venue_points_5,4.74,0.000000,2.60,3.000000,3.00000
away_venue_goals_against_5,4.74,0.000000,3.20,3.400000,4.50000
home_venue_goals_against_5,4.74,0.000000,2.80,2.800000,4.40000
matchup_total_goals_10,2.74,1.000000,3.80,3.937098,4.83333
matchup_away_goals_10,2.74,0.000000,2.30,2.400000,3.55000
matchup_home_goals_10,2.74,0.166667,2.25,2.400000,3.66667
matchup_total_goals_40,2.74,1.000000,3.55,3.675000,4.83333


Baseline features with largest upper-tail jump (max / p995):


,missing_pct,min,p99,p995,max
home_days_since_last_match,0.78,2.000000,91.0000,104.000000,755.00000
away_days_since_last_match,0.80,2.000000,90.0000,104.000000,448.00000
away_goals_for_40,2.38,0.000000,2.6500,2.850000,5.00000
away_venue_goals_for_5,4.74,0.000000,2.8000,3.000000,5.20000
matchup_home_goals_40,2.74,0.166667,2.1125,2.223165,3.66667
home_goals_against_40,2.35,0.000000,2.2750,2.431147,4.00000
away_goals_for_10,2.38,0.000000,2.8000,3.100000,5.00000
home_venue_goals_against_5,4.74,0.000000,2.8000,2.800000,4.40000


#### Shared Logistic Regression preprocessing pipeline

Unlike XGBoost, Logistic Regression cannot handle missing values natively and is sensitive to feature scales and extreme outliers (such as `days_since_last_match`, which can exceed 1,000 days after long absences from the dataset). To prevent leakage and ensure identical preprocessing for both baseline and extended feature sets, we encapsulate all steps in a single `scikit-learn` `Pipeline` fitted strictly on training data:
1. **`QuantileClipper`:** caps extreme values at training-set quantiles (`0.1%` lower, `99.5%` upper).
2. **`SimpleImputer(strategy="median")`:** fills structural missing values (teams with fewer than 3 prior matches) using training medians.
3. **`StandardScaler`:** standardizes all features to zero mean and unit variance so L2 regularization treats features equally.
4. **`LogisticRegression`:** L2-regularized binary classifier with tunable regularization strength `C`.

In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


class QuantileClipper(BaseEstimator, TransformerMixin):
    """Cap extreme feature values using quantiles learned on training data only."""

    def __init__(self, lower_q=0.001, upper_q=0.995):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X_arr = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(X_arr, self.lower_q, axis=0)
        self.upper_ = np.nanquantile(X_arr, self.upper_q, axis=0)
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=float)
        return np.clip(X_arr, self.lower_, self.upper_)


def make_logreg_pipeline(C=0.1, lower_q=0.001, upper_q=0.995):
    """Shared Logistic Regression pipeline (used on baseline and extended features)."""
    return Pipeline([
        ("clip", QuantileClipper(lower_q=lower_q, upper_q=upper_q)),
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(C=C, max_iter=1000)),
    ])

#### Experiment: 2019/20 warm-up season and COVID (`closed_doors`) handling

Before hyperparameter tuning, we test all five valid training-set configurations on the `2324` validation season across both Logistic Regression and a regularized baseline XGBoost:
1. Keep `1920` and keep the `closed_doors` indicator feature.
2. Keep `1920` and drop the `closed_doors` feature.
3. Drop `1920` (use it only as warm-up history) and keep `closed_doors`.
4. Drop `1920` and drop `closed_doors`.
5. Keep `1920` and remove all behind-closed-doors matches (`closed_doors == 1`) from training.

We compare variants primarily by validation **log loss** (and verify consistency with **accuracy**) to decide on a single training setup that both baseline and extended models will use.

In [11]:
from xgboost import XGBClassifier

experiments = [
    ("1. Keep 1920 + keep closed_doors col",  ["1920", "2021", "2122", "2223"], False, True),
    ("2. Keep 1920 + drop closed_doors col",  ["1920", "2021", "2122", "2223"], False, False),
    ("3. Drop 1920 + keep closed_doors col",  ["2021", "2122", "2223"],         False, True),
    ("4. Drop 1920 + drop closed_doors col",  ["2021", "2122", "2223"],         False, False),
    ("5. Keep 1920 + drop closed_doors rows", ["1920", "2021", "2122", "2223"], True,  False),
]

results = []

for label, tr_seasons, drop_cd_rows, use_cd_col in experiments:
    tr, va, _ = split_by_season(
        baseline_data,
        train_seasons=tr_seasons,
        drop_closed_doors=drop_cd_rows,
    )
    feats = baseline_features if use_cd_col else [f for f in baseline_features if f != "closed_doors"]

    X_tr, y_tr = tr[feats], tr[target]
    X_va, y_va = va[feats], va[target]

    # Logistic Regression
    lr = make_logreg_pipeline(C=0.1)
    lr.fit(X_tr, y_tr)
    lr_probs = lr.predict_proba(X_va)[:, 1]
    lr_metrics, _ = evaluate_predictions(va, lr_probs, model_name=f"LogReg | {label}")
    lr_metrics["train_matches"] = len(tr)
    lr_metrics["n_features"] = len(feats)
    lr_metrics["best_iter"] = np.nan
    results.append(lr_metrics)

    # Regularized baseline XGBoost (keeps NaNs natively)
    xgb = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=3,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5.0,
        objective="binary:logistic",
        eval_metric="logloss",
        early_stopping_rounds=40,
        random_state=42,
        n_jobs=-1,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_probs = xgb.predict_proba(X_va)[:, 1]
    xgb_metrics, _ = evaluate_predictions(va, xgb_probs, model_name=f"XGB    | {label}")
    xgb_metrics["train_matches"] = len(tr)
    xgb_metrics["n_features"] = len(feats)
    xgb_metrics["best_iter"] = xgb.best_iteration
    results.append(xgb_metrics)

exp_table = pd.DataFrame(results)[
    ["model", "train_matches", "n_features", "accuracy", "log_loss", "pred_over_rate", "best_iter"]
]
display(exp_table.sort_values("log_loss"))

,model,train_matches,n_features,accuracy,log_loss,pred_over_rate,best_iter
9,XGB | 5. Keep 1920 + drop closed_doors rows,19972,40,0.570738,0.678687,0.503106,137.0
5,XGB | 3. Drop 1920 + keep closed_doors col,21728,41,0.568116,0.678694,0.482540,161.0
1,XGB | 1. Keep 1920 + keep closed_doors col,28181,41,0.569496,0.678888,0.490269,231.0
3,XGB | 2. Keep 1920 + drop closed_doors col,28181,40,0.570186,0.678911,0.489303,170.0
7,XGB | 4. Drop 1920 + drop closed_doors col,21728,40,0.566874,0.678948,0.479917,155.0
4,LogReg | 3. Drop 1920 + keep closed_doors col,21728,41,0.566598,0.679298,0.511663,NaN
6,LogReg | 4. Drop 1920 + drop closed_doors col,21728,40,0.565908,0.679537,0.499379,NaN
0,LogReg | 1. Keep 1920 + keep closed_doors col,28181,41,0.565355,0.679746,0.510145,NaN
2,LogReg | 2. Keep 1920 + drop closed_doors col,28181,40,0.564803,0.679890,0.500207,NaN
8,LogReg | 5. Keep 1920 + drop closed_doors rows,19972,40,0.565908,0.680227,0.513734,NaN


**Decision on `1920` and `closed_doors`:**
* **Keep `closed_doors` as a feature:** Across both Logistic Regression and XGBoost, keeping `closed_doors` outperforms dropping it in every paired comparison on `2324` log loss. It allows the model to absorb the lower COVID scoring environment (`47.6%` over-2.5 rate in `2021`) without biasing predictions downward on normal-crowd seasons (`closed_doors = 0`).
* **Use `1920` as warm-up history only (train on `2021`–`2223`, 21,728 matches):** Dropping `1920` improves validation log loss over keeping `1920` for both Logistic Regression (`0.67930` vs. `0.67975`) and XGBoost (`0.67869` vs. `0.67889`). Although removing COVID rows (`19,972` matches) ties XGBoost log loss within `0.00001`, it produces the worst Logistic Regression log loss (`0.68023`) and retains the noisy `1920` warm-up season.

Therefore, all subsequent baseline and extended models use **train = `["2021", "2122", "2223"]`**, **validation = `["2324"]`**, **test = `["2425"]`**, and **keep all 41 baseline features including `closed_doors`**.

#### Tuning the baseline Logistic Regression benchmark

We lock in the selected chronological split (`2021`–`2223` train, `2324` validation, `2425` held-out test) and assert that no extended or ID columns leak into `baseline_features`. Using our shared `make_logreg_pipeline`, we tune the L2 regularization strength `C` across a logarithmic grid on the `2324` validation season (evaluating both training and validation log loss to verify there is no overfitting) and check whether quantile clipping improves generalization over unclipped features.

In [12]:
# Lock in final split configuration (shared with Person B)
FINAL_TRAIN_SEASONS = ["2021", "2122", "2223"]
FINAL_DROP_CLOSED_DOORS = False

train_base, val_base, test_base = split_by_season(
    baseline_data,
    train_seasons=FINAL_TRAIN_SEASONS,
    val_seasons=VAL_SEASONS,
    test_seasons=TEST_SEASONS,
    drop_closed_doors=FINAL_DROP_CLOSED_DOORS,
)

# Defensive leakage checks
assert set(baseline_features).isdisjoint(extended_features), "Extended feature found in baseline_features!"
assert set(baseline_features).isdisjoint(id_cols), "ID column found in baseline_features!"
assert target not in baseline_features, "Target found in baseline_features!"

X_train_base, y_train_base = train_base[baseline_features], train_base[target]
X_val_base, y_val_base = val_base[baseline_features], val_base[target]

# Tune regularization strength C (and compare clipped vs. unclipped)
lr_rows = []
for clip_label, lq, uq in [("clipped (0.1%-99.5%)", 0.001, 0.995), ("unclipped", 0.0, 1.0)]:
    for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
        pipe = make_logreg_pipeline(C=C, lower_q=lq, upper_q=uq)
        pipe.fit(X_train_base, y_train_base)

        tr_probs = pipe.predict_proba(X_train_base)[:, 1]
        va_probs = pipe.predict_proba(X_val_base)[:, 1]

        va_metrics, _ = evaluate_predictions(val_base, va_probs, model_name=f"LogReg C={C}")
        lr_rows.append({
            "clipping": clip_label,
            "C": C,
            "train_log_loss": log_loss(y_train_base, tr_probs, labels=[0, 1]),
            "val_log_loss": va_metrics["log_loss"],
            "val_accuracy": va_metrics["accuracy"],
            "pred_over_rate": va_metrics["pred_over_rate"],
        })

lr_tuning = pd.DataFrame(lr_rows).sort_values("val_log_loss").reset_index(drop=True)
display(lr_tuning.head(10))

best_lr_row = lr_tuning.iloc[0]
best_lr_C = float(best_lr_row["C"])
best_lr_clip = (0.001, 0.995) if "clipped" in best_lr_row["clipping"] else (0.0, 1.0)

best_baseline_lr = make_logreg_pipeline(C=best_lr_C, lower_q=best_lr_clip[0], upper_q=best_lr_clip[1])
best_baseline_lr.fit(X_train_base, y_train_base)
val_probs_base_lr = best_baseline_lr.predict_proba(X_val_base)[:, 1]
base_lr_val_overall, base_lr_val_country = evaluate_predictions(
    val_base, val_probs_base_lr, model_name=f"Baseline LogReg (C={best_lr_C})"
)
print("\nSelected Baseline LogReg on validation (2324):")
display(pd.DataFrame([base_lr_val_overall]))

,clipping,C,train_log_loss,val_log_loss,val_accuracy,pred_over_rate
0,clipped (0.1%-99.5%),0.003,0.680280,0.679230,0.566322,0.509731
1,clipped (0.1%-99.5%),0.001,0.680469,0.679256,0.566736,0.509041
2,clipped (0.1%-99.5%),0.100,0.680156,0.679298,0.566598,0.511663
3,clipped (0.1%-99.5%),0.010,0.680215,0.679302,0.566736,0.510697
4,clipped (0.1%-99.5%),0.030,0.680183,0.679319,0.567426,0.511663
5,clipped (0.1%-99.5%),1.000,0.680123,0.679319,0.567288,0.512077
6,clipped (0.1%-99.5%),3.000,0.680123,0.679320,0.567426,0.512491
7,clipped (0.1%-99.5%),10.000,0.680122,0.679328,0.567840,0.512629
8,clipped (0.1%-99.5%),0.300,0.680141,0.679350,0.567840,0.513458
9,unclipped,0.003,0.680280,0.679359,0.567978,0.508903



Selected Baseline LogReg on validation (2324):


,model,matches,accuracy,log_loss,pred_over_rate,true_over_rate
0,Baseline LogReg (C=0.003),7245,0.566322,0.67923,0.509731,0.531263


#### Tuning the baseline XGBoost model

Football over/under 2.5 outcomes have a low signal-to-noise ratio, so deep or weakly regularized trees easily overfit training noise. To select hyperparameters without overfitting to the `2324` validation season, we tune XGBoost in two steps:
1. **Stage 1 (Internal chronological CV inside training):** We evaluate tree complexity (`max_depth`, `min_child_weight`) and regularization (`colsample_bytree`, `reg_lambda`, `reg_alpha`) using two expanding/rolling season folds strictly inside training (`train 2021 -> eval 2122`, and `train 2021–2122 -> eval 2223`) so hyperparameter selection does not overfit `2324`.
2. **Stage 2 (Validation check on `2324`):** We refit the top candidate configurations on all training seasons (`2021`–`2223`) and evaluate them on `2324` (checking both train and validation log loss and accuracy) to choose the final baseline XGBoost settings.

In [13]:
from itertools import product

# Stage 1: Chronological folds strictly inside FINAL_TRAIN_SEASONS (no peeking at 2324)
inner_folds = [
    (["2021"], ["2122"]),
    (["2021", "2122"], ["2223"]),
]

param_grid = {
    "max_depth": [2, 3, 4],
    "min_child_weight": [20, 50, 100],
    "colsample_bytree": [0.6, 0.8],
    "reg_lambda": [5.0, 15.0],
    "reg_alpha": [0.0, 2.0],
}

grid_keys = list(param_grid.keys())
stage1_rows = []

for values in product(*param_grid.values()):
    params = dict(zip(grid_keys, values))
    fold_losses = []
    fold_accs = []
    fold_iters = []

    for f_tr_seasons, f_va_seasons in inner_folds:
        f_tr = train_base[train_base["season"].isin(f_tr_seasons)]
        f_va = train_base[train_base["season"].isin(f_va_seasons)]

        model = XGBClassifier(
            n_estimators=800,
            learning_rate=0.02,
            subsample=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            early_stopping_rounds=50,
            random_state=42,
            n_jobs=-1,
            **params,
        )
        model.fit(
            f_tr[baseline_features], f_tr[target],
            eval_set=[(f_va[baseline_features], f_va[target])],
            verbose=False,
        )
        probs = model.predict_proba(f_va[baseline_features])[:, 1]
        fold_losses.append(log_loss(f_va[target], probs, labels=[0, 1]))
        fold_accs.append(accuracy_score(f_va[target], (probs >= 0.5).astype(int)))
        fold_iters.append(model.best_iteration)

    stage1_rows.append({
        **params,
        "cv_log_loss": np.mean(fold_losses),
        "cv_accuracy": np.mean(fold_accs),
        "mean_best_iter": int(round(np.mean(fold_iters))),
    })

stage1_df = pd.DataFrame(stage1_rows).sort_values("cv_log_loss").reset_index(drop=True)
print("Stage 1 — Top 10 candidates by internal chronological CV (2122 & 2223):")
display(stage1_df.head(10))

# Stage 2: Refit top 6 diverse candidates on all of 2021-2223 and evaluate on validation (2324)
# Note: using a fixed n_estimators scaled slightly from inner-CV (no early-stopping peek on 2324)
stage2_rows = []
for i in range(6):
    row = stage1_df.iloc[i]
    params = {k: (int(row[k]) if k in ["max_depth", "min_child_weight"] else float(row[k])) for k in grid_keys}
    # Slightly more trees since full train (2021-2223) is larger than inner folds
    n_est = max(50, int(round(row["mean_best_iter"] * 1.15)))

    model = XGBClassifier(
        n_estimators=n_est,
        learning_rate=0.02,
        subsample=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        **params,
    )
    model.fit(X_train_base, y_train_base, verbose=False)

    tr_probs = model.predict_proba(X_train_base)[:, 1]
    va_probs = model.predict_proba(X_val_base)[:, 1]
    va_metrics, _ = evaluate_predictions(val_base, va_probs, model_name=f"Cand {i+1}")

    stage2_rows.append({
        "candidate": i + 1,
        **params,
        "n_estimators": n_est,
        "cv_log_loss": row["cv_log_loss"],
        "train_log_loss": log_loss(y_train_base, tr_probs, labels=[0, 1]),
        "val_log_loss": va_metrics["log_loss"],
        "val_accuracy": va_metrics["accuracy"],
        "pred_over_rate": va_metrics["pred_over_rate"],
    })

stage2_df = pd.DataFrame(stage2_rows).sort_values("val_log_loss").reset_index(drop=True)
print("\nStage 2 — Top 6 candidates evaluated cleanly on validation season (2324):")
display(stage2_df)

Stage 1 — Top 10 candidates by internal chronological CV (2122 & 2223):


,max_depth,min_child_weight,colsample_bytree,reg_lambda,reg_alpha,cv_log_loss,cv_accuracy,mean_best_iter
0,2,20,0.6,5.0,0.0,0.681654,0.562521,214
1,3,20,0.6,5.0,0.0,0.681660,0.561823,174
2,2,20,0.8,5.0,0.0,0.681674,0.562373,214
3,3,50,0.6,5.0,0.0,0.681713,0.560451,135
4,4,20,0.6,5.0,0.0,0.681737,0.559988,114
5,2,20,0.6,5.0,2.0,0.681760,0.561970,212
6,2,20,0.8,15.0,0.0,0.681791,0.562304,220
7,2,20,0.8,5.0,2.0,0.681792,0.561819,226
8,3,20,0.8,5.0,0.0,0.681801,0.559557,174
9,3,20,0.6,5.0,2.0,0.681816,0.559222,168



Stage 2 — Top 6 candidates evaluated cleanly on validation season (2324):


,candidate,max_depth,min_child_weight,colsample_bytree,reg_lambda,reg_alpha,n_estimators,cv_log_loss,train_log_loss,val_log_loss,val_accuracy,pred_over_rate
0,6,2,20,0.6,5.0,2.0,244,0.681760,0.675984,0.678831,0.570600,0.488889
1,2,3,20,0.6,5.0,0.0,200,0.681660,0.672171,0.678831,0.567426,0.479089
2,1,2,20,0.6,5.0,0.0,246,0.681654,0.675804,0.678834,0.571705,0.487233
3,3,2,20,0.8,5.0,0.0,246,0.681674,0.675661,0.678865,0.572257,0.485024
4,4,3,50,0.6,5.0,0.0,155,0.681713,0.674420,0.679183,0.567702,0.482126
5,5,4,20,0.6,5.0,0.0,131,0.681737,0.670261,0.679400,0.564941,0.471912


**Findings from initial tuning:**
* **Logistic Regression:** Quantile clipping (`0.1%`–`99.5%`) outperformed unclipped features across all tested regularization strengths (`C`). Stronger L2 regularization (`C = 0.003`) achieved the best validation log loss (`0.67923`, accuracy `56.63%`) with no overfitting (`train_log_loss = 0.68028`).
* **XGBoost:** Shallow trees (`max_depth = 2`) generalized best in both internal chronological CV (`2122` & `2223`) and clean validation (`2324`), reaching `0.67883` log loss and `57.06%`–`57.17%` accuracy while maintaining a small train-to-validation gap (`~0.003`). Deeper trees (`max_depth = 3` and `4`) showed clear overfitting (`train_log_loss` down to `0.670`) and worse validation accuracy.
* Because the top candidates sat at the lower boundaries of the initial grid (`max_depth = 2`, `min_child_weight = 20`, `colsample_bytree = 0.6`, `reg_lambda = 5.0`), we run one focused refinement pass around that region and test whether adding `Div` as a categorical baseline feature improves upon `div_goals_365d`.

#### Focused refinement around shallow trees and `Div` categorical check

We run a compact second-stage search around the winning shallow-tree region (`max_depth` in `{1, 2}`, lower `min_child_weight` and `colsample_bytree`, and L1/L2 regularization around `reg_lambda = 5.0`), still selecting candidates via internal chronological CV before validating the top finalists on `2324`.

We also test adding `Div` as a native categorical feature to the best baseline XGBoost configuration (using `enable_categorical=True`) to verify whether explicit league identity adds predictive value beyond `div_goals_365d`. If it does not improve validation log loss, we keep the 41 exported `baseline_features` unchanged so all team members stay on the exact same feature set.

In [14]:
refined_grid = {
    "max_depth": [1, 2],
    "min_child_weight": [10, 20, 35],
    "colsample_bytree": [0.4, 0.6],
    "reg_lambda": [2.0, 5.0, 10.0],
    "reg_alpha": [0.0, 2.0, 5.0],
}

ref_keys = list(refined_grid.keys())
ref_s1_rows = []

for values in product(*refined_grid.values()):
    params = dict(zip(ref_keys, values))
    fold_losses, fold_accs, fold_iters = [], [], []

    for f_tr_seasons, f_va_seasons in inner_folds:
        f_tr = train_base[train_base["season"].isin(f_tr_seasons)]
        f_va = train_base[train_base["season"].isin(f_va_seasons)]

        model = XGBClassifier(
            n_estimators=1000,
            learning_rate=0.02,
            subsample=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            early_stopping_rounds=50,
            random_state=42,
            n_jobs=-1,
            **params,
        )
        model.fit(
            f_tr[baseline_features], f_tr[target],
            eval_set=[(f_va[baseline_features], f_va[target])],
            verbose=False,
        )
        probs = model.predict_proba(f_va[baseline_features])[:, 1]
        fold_losses.append(log_loss(f_va[target], probs, labels=[0, 1]))
        fold_accs.append(accuracy_score(f_va[target], (probs >= 0.5).astype(int)))
        fold_iters.append(model.best_iteration)

    ref_s1_rows.append({
        **params,
        "cv_log_loss": np.mean(fold_losses),
        "cv_accuracy": np.mean(fold_accs),
        "mean_best_iter": int(round(np.mean(fold_iters))),
    })

ref_s1_df = pd.DataFrame(ref_s1_rows).sort_values("cv_log_loss").reset_index(drop=True)
print("Refined Stage 1 — Top 8 candidates by internal chronological CV:")
display(ref_s1_df.head(8))

ref_s2_rows = []
for i in range(6):
    row = ref_s1_df.iloc[i]
    params = {k: (int(row[k]) if k in ["max_depth", "min_child_weight"] else float(row[k])) for k in ref_keys}
    n_est = max(50, int(round(row["mean_best_iter"] * 1.15)))

    model = XGBClassifier(
        n_estimators=n_est,
        learning_rate=0.02,
        subsample=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        **params,
    )
    model.fit(X_train_base, y_train_base, verbose=False)

    tr_probs = model.predict_proba(X_train_base)[:, 1]
    va_probs = model.predict_proba(X_val_base)[:, 1]
    va_metrics, _ = evaluate_predictions(val_base, va_probs, model_name=f"RefCand {i+1}")

    ref_s2_rows.append({
        "candidate": i + 1,
        **params,
        "n_estimators": n_est,
        "cv_log_loss": row["cv_log_loss"],
        "train_log_loss": log_loss(y_train_base, tr_probs, labels=[0, 1]),
        "val_log_loss": va_metrics["log_loss"],
        "val_accuracy": va_metrics["accuracy"],
        "pred_over_rate": va_metrics["pred_over_rate"],
    })

ref_s2_df = pd.DataFrame(ref_s2_rows).sort_values("val_log_loss").reset_index(drop=True)
print("\nRefined Stage 2 — Top 6 candidates evaluated on validation season (2324):")
display(ref_s2_df)

# Optional check: Does adding `Div` as a categorical feature improve the best XGBoost model?
best_ref = ref_s2_df.iloc[0]
best_xgb_params = {
    "n_estimators": int(best_ref["n_estimators"]),
    "learning_rate": 0.02,
    "max_depth": int(best_ref["max_depth"]),
    "min_child_weight": int(best_ref["min_child_weight"]),
    "subsample": 0.8,
    "colsample_bytree": float(best_ref["colsample_bytree"]),
    "reg_lambda": float(best_ref["reg_lambda"]),
    "reg_alpha": float(best_ref["reg_alpha"]),
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
}

div_cats = sorted(train_base["Div"].unique())
X_tr_div = X_train_base.copy()
X_va_div = X_val_base.copy()
X_tr_div["Div"] = pd.Categorical(train_base["Div"], categories=div_cats)
X_va_div["Div"] = pd.Categorical(val_base["Div"], categories=div_cats)

xgb_div = XGBClassifier(enable_categorical=True, **best_xgb_params)
xgb_div.fit(X_tr_div, y_train_base, verbose=False)
div_probs = xgb_div.predict_proba(X_va_div)[:, 1]
div_metrics, _ = evaluate_predictions(val_base, div_probs, model_name="Baseline XGB + Div")

print("\nComparison: 41 baseline features vs. adding Div (validation 2324):")
display(pd.DataFrame([
    {
        "model": "Baseline XGB (41 features)",
        "accuracy": best_ref["val_accuracy"],
        "log_loss": best_ref["val_log_loss"],
        "pred_over_rate": best_ref["pred_over_rate"],
    },
    {
        "model": div_metrics["model"],
        "accuracy": div_metrics["accuracy"],
        "log_loss": div_metrics["log_loss"],
        "pred_over_rate": div_metrics["pred_over_rate"],
    },
]))

Refined Stage 1 — Top 8 candidates by internal chronological CV:


,max_depth,min_child_weight,colsample_bytree,reg_lambda,reg_alpha,cv_log_loss,cv_accuracy,mean_best_iter
0,2,10,0.6,5.0,0.0,0.681555,0.561904,237
1,2,20,0.6,2.0,0.0,0.681636,0.560801,213
2,2,20,0.6,5.0,0.0,0.681654,0.562521,214
3,2,10,0.6,2.0,0.0,0.681662,0.560111,207
4,2,20,0.4,2.0,0.0,0.681667,0.560594,220
5,2,10,0.6,2.0,2.0,0.681694,0.561558,244
6,2,10,0.4,5.0,0.0,0.681698,0.561963,246
7,2,35,0.6,2.0,0.0,0.681706,0.561899,216



Refined Stage 2 — Top 6 candidates evaluated on validation season (2324):


,candidate,max_depth,min_child_weight,colsample_bytree,reg_lambda,reg_alpha,n_estimators,cv_log_loss,train_log_loss,val_log_loss,val_accuracy,pred_over_rate
0,5,2,20,0.4,2.0,0.0,253,0.681667,0.675722,0.678609,0.570738,0.482402
1,1,2,10,0.6,5.0,0.0,273,0.681555,0.675324,0.678732,0.572257,0.486128
2,6,2,10,0.6,2.0,2.0,281,0.681694,0.675238,0.678762,0.569772,0.485576
3,4,2,10,0.6,2.0,0.0,238,0.681662,0.675734,0.678801,0.570186,0.489303
4,2,2,20,0.6,2.0,0.0,245,0.681636,0.675658,0.678830,0.570324,0.487233
5,3,2,20,0.6,5.0,0.0,246,0.681654,0.675804,0.678834,0.571705,0.487233



Comparison: 41 baseline features vs. adding Div (validation 2324):


,model,accuracy,log_loss,pred_over_rate
0,Baseline XGB (41 features),0.570738,0.678609,0.482402
1,Baseline XGB + Div,0.566460,0.678972,0.461560


**Final baseline model decisions:**
* **Selected Baseline XGBoost:** `max_depth=2`, `min_child_weight=20`, `colsample_bytree=0.4`, `subsample=0.8`, `reg_lambda=2.0`, `reg_alpha=0.0`, `learning_rate=0.02`, `n_estimators=253`. It achieves the lowest validation log loss (`0.67861`, accuracy `57.07%`) with minimal overfitting (`train_log_loss = 0.67572`), outperforming deeper trees (`max_depth >= 3`), stumps (`max_depth = 1`), and the tuned Logistic Regression benchmark (`0.67923` log loss, `56.63%` accuracy).
* **Reject `Div` as a feature:** Adding `Div` as a categorical feature degraded both validation log loss (`0.67897` vs. `0.67861`) and accuracy (`56.65%` vs. `57.07%`), confirming that `div_goals_365d` already captures league scoring rates more effectively. We therefore keep the 41 exported `baseline_features` unchanged.

#### Final baseline summary ($A_0$ on validation season `2324`) and prediction export

We wrap the tuned baseline XGBoost and Logistic Regression models into reusable constructors (`make_baseline_xgb` and `make_logreg_pipeline`), report overall and per-country validation accuracy ($A_0$) and log loss on `2324`, and export per-match validation probabilities to `exports/baseline_val_predictions.csv` for downstream pricing checks. The held-out test season (`2425`) remains untouched until both baseline and extended models are finalized for the single test evaluation in Section 6.3.

In [15]:
from pathlib import Path

BASELINE_LR_C = 0.003
BASELINE_XGB_PARAMS = {
    "n_estimators": 253,
    "learning_rate": 0.02,
    "max_depth": 2,
    "min_child_weight": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.4,
    "reg_lambda": 2.0,
    "reg_alpha": 0.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1,
}


def make_baseline_xgb():
    """Return the final tuned baseline XGBoost classifier."""
    return XGBClassifier(**BASELINE_XGB_PARAMS)


# Fit final baseline models on FINAL_TRAIN_SEASONS (2021-2223)
final_base_lr = make_logreg_pipeline(C=BASELINE_LR_C, lower_q=0.001, upper_q=0.995)
final_base_lr.fit(X_train_base, y_train_base)

final_base_xgb = make_baseline_xgb()
final_base_xgb.fit(X_train_base, y_train_base, verbose=False)

# Predict on validation season (2324) only — test season (2425) stays untouched
val_probs_base_lr = final_base_lr.predict_proba(X_val_base)[:, 1]
val_probs_base_xgb = final_base_xgb.predict_proba(X_val_base)[:, 1]

# Sanity checks on predicted probabilities
for name, probs in [("Baseline LogReg", val_probs_base_lr), ("Baseline XGB", val_probs_base_xgb)]:
    assert not np.isnan(probs).any(), f"{name} produced NaN probabilities!"
    assert ((probs > 0) & (probs < 1)).all(), f"{name} probabilities out of (0, 1)!"

# Overall validation summary (Naive vs. Baseline LogReg vs. Baseline XGB)
p_prior_final = np.full(len(val_base), y_train_base.mean())
naive_ov, _ = evaluate_predictions(val_base, p_prior_final, model_name="Naive (2021-2223 prior)")
lr_ov, lr_by_country = evaluate_predictions(val_base, val_probs_base_lr, model_name="Baseline LogReg (C=0.003)")
xgb_ov, xgb_by_country = evaluate_predictions(val_base, val_probs_base_xgb, model_name="Baseline XGB (tuned)")

print("Overall validation results (season 2324) — Baseline A0:")
display(pd.DataFrame([naive_ov, lr_ov, xgb_ov]))

# Per-country validation A0 comparison
val_country_summary = (
    xgb_by_country.rename(columns={"accuracy": "xgb_acc_A0", "log_loss": "xgb_logloss"})
    .merge(
        lr_by_country[["country", "accuracy", "log_loss"]].rename(
            columns={"accuracy": "logreg_acc_A0", "log_loss": "logreg_logloss"}
        ),
        on="country",
    )
)
val_country_summary["xgb_minus_logreg_pp"] = (
    (val_country_summary["xgb_acc_A0"] - val_country_summary["logreg_acc_A0"]) * 100
).round(2)
print("\nPer-country validation results (season 2324):")
display(val_country_summary)

# Save per-match validation predictions for Person C (exports/ is git-ignored)
baseline_val_preds = build_predictions_df(
    val_base,
    {
        "prob_baseline_xgb": val_probs_base_xgb,
        "prob_baseline_logreg": val_probs_base_lr,
    },
)
out_path = Path("exports/baseline_val_predictions.csv")
baseline_val_preds.to_csv(out_path, index=False)
print(f"\nSaved {len(baseline_val_preds)} validation predictions to {out_path}")
display(baseline_val_preds.head())

Overall validation results (season 2324) — Baseline A0:


,model,matches,accuracy,log_loss,pred_over_rate,true_over_rate
0,Naive (2021-2223 prior),7245,0.468737,0.695440,0.000000,0.531263
1,Baseline LogReg (C=0.003),7245,0.566322,0.679230,0.509731,0.531263
2,Baseline XGB (tuned),7245,0.570738,0.678609,0.482402,0.531263



Per-country validation results (season 2324):


,country,matches,xgb_acc_A0,xgb_logloss,logreg_acc_A0,logreg_logloss,xgb_minus_logreg_pp
0,england,2036,0.555992,0.685527,0.551081,0.685625,0.49
1,spain,842,0.618765,0.666333,0.631829,0.665328,-1.31
2,scotland,768,0.557292,0.679714,0.554688,0.682153,0.26
3,italy,760,0.543421,0.686529,0.530263,0.688755,1.32
4,france,685,0.545985,0.689954,0.541606,0.690908,0.44
5,germany,612,0.616013,0.664664,0.601307,0.663363,1.47
6,turkey,379,0.527704,0.680633,0.519789,0.683495,0.79
7,belgium,312,0.576923,0.680609,0.560897,0.679195,1.60
8,netherlands,306,0.620915,0.651133,0.617647,0.654269,0.33
9,portugal,306,0.604575,0.671076,0.594771,0.672214,0.98



Saved 7245 validation predictions to exports\baseline_val_predictions.csv


,country,season,Date,HomeTeam,AwayTeam,over_2_5,prob_baseline_xgb,prob_baseline_logreg
1146,belgium,2324,2023-07-28,St. Gilloise,Anderlecht,0,0.586099,0.528878
1147,belgium,2324,2023-07-29,Eupen,Westerlo,1,0.564085,0.535954
1148,belgium,2324,2023-07-29,Charleroi,Oud-Heverlee Leuven,0,0.534940,0.509606
1149,belgium,2324,2023-07-29,RWD Molenbeek,Genk,1,0.446231,0.486126
1150,belgium,2324,2023-07-30,Antwerp,Cercle Brugge,0,0.542434,0.521619


**Section 6.1 summary (validation season `2324`, 7,245 matches):**
* Both baseline models beat the naive training-prior floor (`46.87%` accuracy, `0.69544` log loss) by roughly 10 percentage points, while remaining below the `57.4%` market-odds reference as expected without betting odds.
* **Tuned Baseline XGBoost** achieves **`57.07%`** overall validation accuracy ($A_0$) and **`0.67861`** log loss, improving over the **Baseline Logistic Regression** benchmark (`56.63%`, `0.67923`) by `+0.44` percentage points overall and outperforming it on accuracy in 9 of the 11 countries.
* Predicted over-2.5 rates (`48.24%` for XGBoost, `50.97%` for Logistic Regression) reflect the lower historical scoring base rate of `2021`–`2223` rather than `2324`'s unusually high `53.13%` rate, consistent with a leak-free chronological split.
* Per-country validation accuracies range from `52.77%` (Turkey) to `62.09%` (Netherlands), reflecting both genuine differences in league scoring levels and small-sample noise in markets with ~240–380 matches. Validation predictions are saved to `exports/baseline_val_predictions.csv`; the test season (`2425`) remains untouched for Section 6.3.